In [10]:
# ============================================================
# 3D V-NET TRAINING - TASK03 LIVER
# Use with the previous corrected Dataset
# ============================================================


import torch
import torch.nn as nn
from tqdm import tqdm
import numpy as np



# ============================
# V-NET BLOCK
# ============================


class VNetBlock(nn.Module):

    def __init__(self,in_c,out_c):

        super().__init__()

        self.block=nn.Sequential(

            nn.Conv3d(
                in_c,
                out_c,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm3d(out_c),

            nn.PReLU(),


            nn.Conv3d(
                out_c,
                out_c,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm3d(out_c),

            nn.PReLU()
        )


    def forward(self,x):

        return self.block(x)




# ============================
# 3D V-NET
# ============================


class VNet3D(nn.Module):

    def __init__(self):

        super().__init__()



        # Encoder

        self.enc1=VNetBlock(
            1,
            16
        )


        self.down1=nn.Conv3d(
            16,
            32,
            2,
            stride=2
        )



        self.enc2=VNetBlock(
            32,
            32
        )


        self.down2=nn.Conv3d(
            32,
            64,
            2,
            stride=2
        )



        self.enc3=VNetBlock(
            64,
            64
        )


        self.down3=nn.Conv3d(
            64,
            128,
            2,
            stride=2
        )



        # Bottleneck

        self.bridge=VNetBlock(
            128,
            128
        )



        # Decoder


        self.up3=nn.ConvTranspose3d(
            128,
            64,
            2,
            stride=2
        )


        self.dec3=VNetBlock(
            128,
            64
        )



        self.up2=nn.ConvTranspose3d(
            64,
            32,
            2,
            stride=2
        )


        self.dec2=VNetBlock(
            64,
            32
        )



        self.up1=nn.ConvTranspose3d(
            32,
            16,
            2,
            stride=2
        )


        self.dec1=VNetBlock(
            32,
            16
        )



        self.out=nn.Conv3d(
            16,
            1,
            kernel_size=1
        )




    def forward(self,x):


        e1=self.enc1(x)


        e2=self.enc2(
            self.down1(e1)
        )


        e3=self.enc3(
            self.down2(e2)
        )


        b=self.bridge(
            self.down3(e3)
        )



        d3=self.up3(b)


        d3=torch.cat(
            [
                d3,
                e3
            ],
            dim=1
        )


        d3=self.dec3(d3)




        d2=self.up2(d3)


        d2=torch.cat(
            [
                d2,
                e2
            ],
            dim=1
        )


        d2=self.dec2(d2)



        d1=self.up1(d2)


        d1=torch.cat(
            [
                d1,
                e1
            ],
            dim=1
        )


        d1=self.dec1(d1)



        return torch.sigmoid(
            self.out(d1)
        )




# ============================
# CREATE MODEL
# ============================


device="cuda" if torch.cuda.is_available() else "cpu"


model=VNet3D().to(device)


optimizer=torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)



# ============================
# TRAINING
# ============================


EPOCHS=5


for epoch in range(EPOCHS):


    model.train()


    total_loss=0


    loop=tqdm(train_loader)



    for img,mask in loop:


        img=img.to(device)

        mask=mask.to(device)



        pred=model(img)



        loss=dice_loss(
            pred,
            mask
        )



        optimizer.zero_grad()

        loss.backward()

        optimizer.step()



        total_loss+=loss.item()



        loop.set_description(
            f"Epoch {epoch+1}/{EPOCHS}"
        )



    print(
        "Loss:",
        total_loss/len(train_loader)
    )




# ============================
# VALIDATION DICE
# ============================


model.eval()


dice_scores=[]



with torch.no_grad():


    for img,mask in val_loader:


        img=img.to(device)

        mask=mask.to(device)



        pred=model(img)


        pred=(pred>0.5).float()



        intersection=(pred*mask).sum()



        dice=(2*intersection)/(
            pred.sum()+mask.sum()+1e-5
        )



        dice_scores.append(
            dice.item()
        )



print(
    "V-Net Validation Dice:",
    np.mean(dice_scores)
)



# SAVE

torch.save(
    model.state_dict(),
    "VNet3D_Liver.pth"
)


print(
    "Saved: VNet3D_Liver.pth"
)

Epoch 1/5: 100%|██████████| 104/104 [02:15<00:00,  1.31s/it]


Loss: 0.8859317801319636


Epoch 2/5: 100%|██████████| 104/104 [02:15<00:00,  1.31s/it]


Loss: 0.8701589692097443


Epoch 3/5: 100%|██████████| 104/104 [02:15<00:00,  1.30s/it]


Loss: 0.8640225558326795


Epoch 4/5: 100%|██████████| 104/104 [02:12<00:00,  1.28s/it]


Loss: 0.85776103918369


Epoch 5/5: 100%|██████████| 104/104 [02:12<00:00,  1.27s/it]


Loss: 0.8508305257329574
V-Net Validation Dice: 0.8517440557479858
Saved: VNet3D_Liver.pth


In [2]:
import os

print("Images:")
print(sorted(os.listdir(IMG_DIR))[:10])

print("\nMasks:")
print(sorted(os.listdir(MASK_DIR))[:10])

Images:
['liver_0_img.npy', 'liver_100_img.npy', 'liver_101_img.npy', 'liver_102_img.npy', 'liver_103_img.npy', 'liver_104_img.npy', 'liver_105_img.npy', 'liver_106_img.npy', 'liver_107_img.npy', 'liver_108_img.npy']

Masks:
['liver_0_liverMask.npy', 'liver_100_liverMask.npy', 'liver_101_liverMask.npy', 'liver_102_liverMask.npy', 'liver_103_liverMask.npy', 'liver_104_liverMask.npy', 'liver_105_liverMask.npy', 'liver_106_liverMask.npy', 'liver_107_liverMask.npy', 'liver_108_liverMask.npy']


In [3]:
DATA_DIR = "/kaggle/input/liver-cancer-multiclass-dataset/Liver_Dataset/Liver_Dataset"